In [4]:
import polars as pl
import numpy as np
import time
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models import Phrases
from gensim.models.phrases import Phraser
from gensim.parsing.preprocessing import STOPWORDS
from gensim.corpora import MmCorpus
import joblib
from tqdm.notebook import tqdm
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
import gc

yelp_stopwords = [
    "ive", "im", "us", "got", "went", "came",
    "also", "even", "really", "just", "would",
    "always", "much", "one", "get", "go", "back"
]
stopwords_list = list(STOPWORDS) + [""] + yelp_stopwords

csv_reviews_with_categories = '../data/csv/yelp_reviews_with_business.csv'

CATEGORIES = {
    "fast_food":   "Fast Food",
    "steakhouses": "Steakhouses",
    "burguers":     "Burgers",
    "pubs":        "Pubs",
    "mexican":     "Mexican",
}

In [5]:
CHUNK_SIZE = 100_000 

def _stream_tokens(input_csv: str, label: str, stopwords_list: list, bigram_model=None):
    """Generator: yields one tokenized doc at a time, chunk by chunk."""
    offset = 0
    while True:
        chunk = (
            pl.scan_csv(input_csv)
            .filter(pl.col('categories').str.contains(label))
            .slice(offset, CHUNK_SIZE)
            .with_columns(
                pl.col('text')
                .str.replace_all(r'[^a-zA-Z\s]', '')
                .str.to_lowercase()
                .str.split(' ')
                .list.set_difference(stopwords_list)
                .alias('tokens')
            )
            .select('tokens')
            .collect()
        )
        if len(chunk) == 0:
            break

        tokens_list = chunk['tokens'].to_list()
        del chunk

        for tokens in tokens_list:
            yield bigram_model[tokens] if bigram_model else tokens

        offset += CHUNK_SIZE


def train_all_lda(input_csv: str, categories: dict):
    for key, label in categories.items():
        print(f'\n{"="*50}\n[{label}] Entrenando LDA...\n{"="*50}')
        train_lda_streaming(input_csv, key, label)


def train_lda_streaming(input_csv: str, category_key: str, category_label: str):
    model_dir = f'../data/models/{category_key}/lda'
    os.makedirs(model_dir, exist_ok=True)

    bigram_detector = Phrases(min_count=10, threshold=20)

    for tokens in _stream_tokens(input_csv, category_label, stopwords_list):
        bigram_detector.add_vocab([tokens])

    bigram_model = Phraser(bigram_detector)
    del bigram_detector
    bigram_model.save(f'{model_dir}/bigram_model.pkl')

    dictionary = Dictionary()

    buffer = []
    for tokens in _stream_tokens(input_csv, category_label, stopwords_list, bigram_model):
        buffer.append(tokens)
        if len(buffer) >= CHUNK_SIZE:
            dictionary.add_documents(buffer)
            buffer.clear()
    if buffer:
        dictionary.add_documents(buffer)
        buffer.clear()

    dictionary.filter_extremes(no_below=20, no_above=0.5, keep_n=40_000)
    dictionary.save(f'{model_dir}/dictionary.dict')
    print(f'[{category_label}] Diccionario: {len(dictionary):,} tokens')

    corpus_path = f'{model_dir}/corpus.mm'

    def bow_generator():
        for tokens in _stream_tokens(input_csv, category_label, stopwords_list, bigram_model):
            yield dictionary.doc2bow(tokens)

    MmCorpus.serialize(corpus_path, bow_generator())
    corpus = MmCorpus(corpus_path)
    print(f'[{category_label}] Corpus: {corpus.num_docs:,} docs')

    print(f'[{category_label}] Entrenando LDA')
    lda = LdaMulticore(
        corpus=corpus,
        num_topics=10,
        id2word=dictionary,
        workers=max(1, os.cpu_count() - 1),
        passes=3,
        chunksize=2_000,
        random_state=42,
    )
    lda.save(f'{model_dir}/lda_model.gensim')

    os.remove(corpus_path)
    print(f'[{category_label}] LDA guardado: {model_dir}')

In [3]:
def train_all_bertopic(input_csv: str, categories: dict):
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

    for key, label in categories.items():
        model_dir = f'../data/models/{key}/bertopic'
        os.makedirs(model_dir, exist_ok=True)

        df = (
            pl.scan_csv(input_csv)
            .filter(pl.col('categories').str.contains(label))
            .collect()
        )
        sample_size = min(300_000, len(df))
        docs = df.sample(n=sample_size, seed=42)['text'].to_list()
        del df

        print(f'[{label}] Embedding {len(docs):,} docs...')
        embeddings = embedding_model.encode(
            docs,
            batch_size=512,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

        min_df = 10 if sample_size > 100_000 else 3
        vectorizer_model = CountVectorizer(
            stop_words=stopwords_list,
            ngram_range=(2, 3),
            min_df=min_df,
        )
        topic_model = BERTopic(
            calculate_probabilities=False,
            verbose=True,
            vectorizer_model=vectorizer_model,
            nr_topics=30
        )
        topic_model.fit_transform(docs, embeddings=embeddings)
        topic_model.save(f'{model_dir}/bertopic_model.pkl', serialization='pickle')
        print(f'[{label}] BERTopic guardado -> {model_dir}')

        del embeddings, docs, topic_model

    del embedding_model

In [6]:
train_all_lda(csv_reviews_with_categories, CATEGORIES)
#train_all_bertopic(csv_reviews_with_categories, CATEGORIES)


[Fast Food] Entrenando LDA...
[Fast Food] Diccionario: 14,845 tokens
[Fast Food] Corpus: 233,008 docs
[Fast Food] Entrenando LDA
[Fast Food] LDA guardado: ../data/models/fast_food/lda

[Steakhouses] Entrenando LDA...
[Steakhouses] Diccionario: 17,647 tokens
[Steakhouses] Corpus: 240,040 docs
[Steakhouses] Entrenando LDA
[Steakhouses] LDA guardado: ../data/models/steakhouses/lda

[Burgers] Entrenando LDA...
[Burgers] Diccionario: 23,166 tokens
[Burgers] Corpus: 445,895 docs
[Burgers] Entrenando LDA
[Burgers] LDA guardado: ../data/models/burguers/lda

[Pubs] Entrenando LDA...
[Pubs] Diccionario: 17,082 tokens
[Pubs] Corpus: 218,891 docs
[Pubs] Entrenando LDA
[Pubs] LDA guardado: ../data/models/pubs/lda

[Mexican] Entrenando LDA...
[Mexican] Diccionario: 21,489 tokens
[Mexican] Corpus: 432,248 docs
[Mexican] Entrenando LDA
[Mexican] LDA guardado: ../data/models/mexican/lda


In [1]:
def load_category_models(category_key, embedding_model):
    lda_dir = f'../data/models/{category_key}/lda'
    bert_dir = f'../data/models/{category_key}/bertopic'
    
    lda = LdaMulticore.load(f'{lda_dir}/lda_model.gensim')
    dictionary = Dictionary.load(f'{lda_dir}/dictionary.dict')
    bigram_model = Phraser.load(f'{lda_dir}/bigram_model.pkl')
    
    bertopic = BERTopic.load(f'{bert_dir}/bertopic_model.pkl', embedding_model=embedding_model)
    
    lda_topic_words = {}
    for i in range(lda.num_topics):
        words = lda.show_topic(i, topn=3)
        lda_topic_words[i] = f'{words[0][0]}_{words[1][0]}_{words[2][0]}'
        
    bert_labels = bertopic.topic_labels_
    
    return lda, dictionary, bigram_model, bertopic, lda_topic_words, bert_labels

def run_inference_all_categories(input_csv: str, out_dir: str, categories: dict):
    os.makedirs(out_dir, exist_ok=True)

    print('\nCargando SentenceTransformer base para inferencia (se reutiliza)...')
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

    start_time = time.time()

    for key, label in categories.items():
        print(f'\n[{label}] Procesando categoría')
        
        lda, dictionary, bigram_model, bertopic, lda_topic_words, bert_labels = load_category_models(key, embedding_model)

        df = (
            pl.scan_csv(input_csv)
            .filter(pl.col('categories').str.contains(label))
            .collect()
        )
        print(f'[{label}] {len(df):,} reviews cargadas')

        print(f'[{label}] LDA sobre {len(df):,} docs...')

        df_tok = df.with_columns(
            pl.col('text')
            .str.replace_all(r'[^a-zA-Z\s]', '')
            .str.to_lowercase()
            .str.split(' ')
            .list.set_difference(stopwords_list)
            .alias('tokens')
        )
        bigramed = [bigram_model[doc] for doc in df_tok['tokens'].to_list()]
        del df_tok

        bow   = [dictionary.doc2bow(t) for t in bigramed]
        dists = lda[bow]
        del bigramed, bow
    
        dominant_topics, topic_probs = [], []
        for dist in dists:
            if dist:
                best_id, best_prob = max(dist, key=lambda x: x[1])
                dominant_topics.append(lda_topic_words[best_id])
                topic_probs.append(best_prob)
            else:
                dominant_topics.append('none')
                topic_probs.append(0.0)
        del dists

        print(f'[{label}] BERTopic transform sobre {len(df):,} docs')
        bert_topics, _ = bertopic.transform(df['text'].to_list())
        bert_labels_col = [bert_labels.get(t, f'topic_{t}') for t in bert_topics]

        out_path = os.path.join(out_dir, f'yelp_topics_{key}.csv')
        df.with_columns([
            pl.Series('lda_dominant_topic',      dominant_topics),
            pl.Series('lda_topic_probability',   topic_probs),
            pl.Series('bertopic_topic',           bert_topics),
            pl.Series('bertopic_dominant_topic',  bert_labels_col),
        ]).write_csv(out_path)

        print(f'[{label}]Guardado en {out_path}')

        del df, dominant_topics, topic_probs, bert_topics, bert_labels_col
        del lda, dictionary, bigram_model, bertopic, lda_topic_words, bert_labels
        
        gc.collect()

    print(f'\nInferencia completada en {(time.time() - start_time) / 60:.2f} min')

In [5]:
csv_reviews_with_categories = '../data/csv/yelp_reviews_with_business.csv'
csv_output_dir = '../results/topic_modeling'

run_inference_all_categories(csv_reviews_with_categories, csv_output_dir, CATEGORIES)


Cargando SentenceTransformer base para inferencia (se reutiliza)...


Loading weights: 100%|█████████████████████████████████████| 103/103 [00:00<00:00, 764.70it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[Fast Food] Procesando categoría
[Fast Food] 233,008 reviews cargadas
[Fast Food] LDA sobre 233,008 docs...
[Fast Food] BERTopic transform sobre 233,008 docs


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████| 7282/7282 [03:08<00:00, 38.66it/s]
2026-05-03 02:27:03,548 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-03 02:28:50,135 - BERTopic - Dimensionality - Completed ✓
2026-05-03 02:28:50,135 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-03 02:29:36,712 - BERTopic - Cluster - Completed ✓


[Fast Food]Guardado en ../results/topic_modeling/yelp_topics_fast_food.csv

[Steakhouses] Procesando categoría
[Steakhouses] 240,040 reviews cargadas
[Steakhouses] LDA sobre 240,040 docs...
[Steakhouses] BERTopic transform sobre 240,040 docs


...

[Burgers]Guardado en ../results/topic_modeling/yelp_topics_burguers.csv

[Pubs] Procesando categoría
[Pubs] 218,891 reviews cargadas
[Pubs] LDA sobre 218,891 docs...
[Pubs] BERTopic transform sobre 218,891 docs


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████| 6841/6841 [03:06<00:00, 36.62it/s]
2026-05-03 03:04:03,059 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-03 03:06:31,748 - BERTopic - Dimensionality - Completed ✓
2026-05-03 03:06:31,749 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-03 03:07:23,535 - BERTopic - Cluster - Completed ✓


[Pubs]Guardado en ../results/topic_modeling/yelp_topics_pubs.csv

[Mexican] Procesando categoría
[Mexican] 432,248 reviews cargadas
[Mexican] LDA sobre 432,248 docs...
[Mexican] BERTopic transform sobre 432,248 docs


Batches: 100%|███████████████████████████████████████████████████████████████████████████████████| 13508/13508 [05:56<00:00, 37.87it/s]
2026-05-03 03:16:41,638 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-05-03 03:22:01,051 - BERTopic - Dimensionality - Completed ✓
2026-05-03 03:22:01,071 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-05-03 03:24:41,468 - BERTopic - Cluster - Completed ✓


[Mexican]Guardado en ../results/topic_modeling/yelp_topics_mexican.csv

Inferencia completada en 62.81 min
